# Chapter 7: computational activity

EECE 5610 lecture notes, prepared and revised by Milad Siami.

Work the activity at the end of the chapter on paper first. State your prediction and assumptions before running the experiment.

Run the setup cells, inspect the named case function, and change one parameter at a time. All experiments are mathematical simulations; no hardware is connected.

In [ ]:
%matplotlib inline

In [ ]:
# In the browser (JupyterLite), hide a harmless Matplotlib warning from its in-browser backend.
import sys, warnings
if "pyodide" in sys.modules:
    warnings.filterwarnings("ignore", message=r"The \w+ parameter as float was deprecated")

In [ ]:
from pathlib import Path
import argparse,json,math,os
os.environ.setdefault('MPLBACKEND','Agg')
import numpy as np
from scipy import signal,integrate,linalg
import matplotlib

In [ ]:
# Use the notebook plotting backend.
import matplotlib.pyplot as plt

In [ ]:
OUT=Path.cwd()/'activity_figures'
OUT.mkdir(parents=True,exist_ok=True)
plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'figure.dpi':140})
RESULTS={}

In [ ]:
def finish(n,fig,data):
 fig.tight_layout();fig.savefig(OUT/f'case{n:02}.pdf');fig.savefig(OUT/f'case{n:02}.png',dpi=170);plt.close(fig)
 RESULTS[f'{n:02}']=data;print(f'Chapter {n:02}: '+json.dumps(data));return data

In [ ]:
def case01(K=2.0,T_values=(0.1,0.4,0.65)):
 fig,ax=plt.subplots(figsize=(7,2.6));records=[]
 for T in T_values:
  p=math.exp(-2*T);q=p-K*(1-p);k=np.arange(17);y=K/(1+K)*(1-q**k)
  # Independently integrate each held-command interval.
  numeric=[0.0]
  for _ in k[1:]:
   last=numeric[-1];u=K*(1-last)
   numeric.append(integrate.solve_ivp(lambda t,x:-2*x+2*u,(0,T),[last],rtol=1e-10,atol=1e-12).y[0,-1])
  assert np.allclose(numeric,y,rtol=1e-7,atol=1e-7)
  ax.plot(k*T,y,'o-',ms=3,label=f'T={T:g}, pole={q:.3f}');records.append({'T':T,'pole':q,'stable':bool(abs(q)<1)})
 ax.axhline(K/(1+K),color='black',ls=':',label='steady output');ax.set(xlabel='Time (s)',ylabel='Normalized brightness');ax.legend(fontsize=8,ncol=2)
 return finish(1,fig,{'runs':records,'T_limit':math.log((K+1)/(K-1))/2 if K>1 else None,'steady_output':K/(1+K)})

In [ ]:
def case02():
 r=np.linspace(0,2,41);g,h,c,f=.5,1.2,.2,.1;den=1+g*h*c
 y=(h*(g+f)*r+.15)/den
 # Solve the three node equations directly.
 A=np.array([[1,0,c],[-g,1,0],[0,-h,1.]])
 sol=np.linalg.solve(A,np.vstack([r,f*r,np.full_like(r,.15)]))
 assert np.allclose(sol[2],y)
 fig,ax=plt.subplots(figsize=(7,2.6));ax.plot(r,y,label='output disturbance d=0.15');ax.plot(r,h*(g+f)*r/den,'--',label='d=0');ax.set(xlabel='Reference r',ylabel='Output y');ax.legend()
 return finish(2,fig,{'reference_gain':h*(g+f)/den,'output_disturbance_gain':1/den,'disturbance_before_final_gain':h/den})

In [ ]:
def case03(a=.6):
 k=np.arange(21);y=signal.lfilter([1-a],[1,-a],np.ones(20));y=np.r_[0,y]
 assert np.allclose(y,1-a**k)
 # Evaluate the unilateral transform both by a finite series and its closed form.
 z=2.;xs=sum((1-a**j)*z**(-j) for j in range(300));xf=(1-a)*z/((z-1)*(z-a));assert abs(xs-xf)<1e-12
 fig,ax=plt.subplots(figsize=(7,2.6));ax.stem(k,y,basefmt=' ');ax.axhline(1,color='black',ls=':');ax.set(xlabel='Sample k',ylabel='Normalized temperature')
 return finish(3,fig,{'a':a,'first_samples':y[:5].tolist(),'transform_at_z2':xf,'final_value_valid':bool(abs(a)<1)})

In [ ]:
def case04(a=.4,initial=0.):
 u=np.r_[1.,1.,np.zeros(18)];y=[];prev=initial
 for val in u:prev=a*prev+val;y.append(prev)
 expected=np.r_[1.,1.4*a**np.arange(19)] if initial==0 and a==.4 else None
 if expected is not None:assert np.allclose(y,expected)
 yf,_=signal.lfilter([1],[1,-a],u,zi=[a*initial]);assert np.allclose(y,yf)
 fig,ax=plt.subplots(figsize=(7,2.6));ax.stem(np.arange(len(y)),y,basefmt=' ');ax.plot(np.arange(len(u)),u,'s--',label='input');ax.set(xlabel='Sample k',ylabel='Concentration model output');ax.legend()
 energy=y[0]**2+y[1]**2/(1-a*a)
 return finish(4,fig,{'first_samples':y[:5],'energy_infinite':energy})

In [ ]:
def case05(a=.7):
 u=np.r_[np.ones(8),np.zeros(12)];x=0.;w=0.;y=[];yw=[]
 for uk in u:
  y.append(a*x+(1-a)*uk);yw.append((a/10)*w+(1-a)*uk)
  x=a*x+(1-a)*uk;w=a*w+10*(1-a)*uk
 assert np.allclose(y,yw);assert np.allclose(y,signal.lfilter([1-a],[1,-a],u))
 fig,ax=plt.subplots(figsize=(7,2.6));ax.step(np.arange(len(u)),u,where='post',label='input');ax.plot(y,'o-',label='one stored state');ax.plot(yw,'x',label='rescaled state');ax.set(xlabel='Sample k',ylabel='Signal');ax.legend(fontsize=9)
 return finish(5,fig,{'feedthrough':1-a,'impulse_first_value':1-a,'maximum_realization_difference':float(np.max(np.abs(np.array(y)-yw)))})

In [ ]:
def case06(fs=12.,f_low=2.,f_high=10.):
 k=np.arange(13);ts=k/fs;t=np.linspace(0,1,1001)
 s1=np.cos(2*np.pi*f_low*ts);s2=np.cos(2*np.pi*f_high*ts)
 alias=any(abs(v-round(v))<1e-12 for v in [(f_high-f_low)/fs,(f_high+f_low)/fs])
 assert bool(np.allclose(s1,s2,atol=1e-12))==alias
 fig,ax=plt.subplots(figsize=(7,2.6));ax.plot(t,np.cos(2*np.pi*f_low*t),label=f'{f_low:g} Hz');ax.plot(t,np.cos(2*np.pi*f_high*t),alpha=.6,label=f'{f_high:g} Hz');ax.plot(ts,s1,'ko',ms=4,label='shared samples' if alias else f'{f_low:g} Hz samples');
 if not alias:ax.plot(ts,s2,'x',color='gray',label=f'{f_high:g} Hz samples')
 ax.set(xlabel='Time (s)',ylabel='Signal');ax.legend(fontsize=9,ncol=3)
 return finish(6,fig,{'fs':fs,'samples_identical':alias,'max_sample_difference':float(np.max(abs(s1-s2))),'values_at_1_over_48':[math.cos(2*math.pi*f/48) for f in [f_low,f_high]]})

In [ ]:
def case07(T=.25,k=2):
 t0=k*T;t=np.linspace(t0,t0+T,200);tau=t-t0
 true=t*t;zoh=np.full_like(t,t0*t0);slope=(t0*t0-(t0-T)**2)/T;foh=t0*t0+slope*tau
 assert np.allclose(true-foh,tau*(T+tau))
 mid=t0+T/2;interp=(t0*t0+(t0+T)**2)/2
 fig,ax=plt.subplots(figsize=(7,2.6));ax.plot(t,true,label='true quadratic');ax.plot(t,zoh,label='ZOH');ax.plot(t,foh,label='causal slope estimate');ax.plot([t0,t0+T],[t0*t0,(t0+T)**2],':',label='future sample interpolation');ax.set(xlabel='Time (s)',ylabel='Signal');ax.legend(fontsize=8,ncol=2)
 return finish(7,fig,{'midpoint':mid,'true':mid*mid,'zoh':t0*t0,'causal_FOH':t0*t0+slope*T/2,'future_interpolation':interp})

In [ ]:
def case08(T=math.log(2)):
 p=math.exp(-T);b1=1-(1+T)*p;b0=p*p+(T-1)*p;k=np.arange(13)
 A=np.array([[-1.,0.],[1.,-1.]]);B=np.array([[1.],[0.]]);C=np.array([[0.,1.]])
 Ad,Bd,_,_,_=signal.cont2discrete((A,B,C,np.zeros((1,1))),T)
 x=np.zeros(2);ya=[0.]
 for _ in k[1:]:x=Ad@x+Bd[:,0];ya.append(float((C@x)[0]))
 exact=1-(1+k*T)*p**k;assert np.allclose(ya,exact)
 _,yd=signal.dstep(([b1,b0],[1,-2*p,p*p],T),n=len(k));assert np.allclose(yd[0].ravel(),exact)
 _,intermediate=signal.dstep(([(1-p)**2],[1,-2*p,p*p],T),n=len(k));sep=intermediate[0].ravel()
 fig,ax=plt.subplots(figsize=(7,2.6));ax.plot(k,exact,'o-',label='continuous cascade, one input hold');ax.plot(k,sep,'s--',label='sampler and hold added between stages');ax.set(xlabel='Sample k',ylabel='Step output');ax.legend(fontsize=8)
 return finish(8,fig,{'T':T,'p':p,'numerator':[b1,b0],'one_hold_first_samples':exact[:3].tolist(),'intermediate_hold_first_samples':sep[:3].tolist()})

In [ ]:
def case09(T=.4,tau=.12):
 assert 0<=tau<T;p=math.exp(-T);bn=1-math.exp(-(T-tau));bo=math.exp(-(T-tau))-p
 y=[0.];prev=0.
 for k in range(16):
  uk=1.;ymid=math.exp(-tau)*y[-1]+(1-math.exp(-tau))*prev
  yn=math.exp(-(T-tau))*ymid+(1-math.exp(-(T-tau)))*uk
  assert abs(yn-(p*y[-1]+bo*prev+bn*uk))<1e-12
  y.append(yn);prev=uk
 fig,ax=plt.subplots(figsize=(7,2.6));k=np.arange(len(y));ax.plot(k*T,y,'o-',label=f'latency {tau:g} s');ax.plot(k*T,1-p**k,'s--',label='zero latency');ax.set(xlabel='Time (s)',ylabel='Lag output');ax.legend()
 return finish(9,fig,{'T':T,'tau':tau,'p':p,'b_new':bn,'b_old':bo,'dc_gain':(bn+bo)/(1-p),'first_step_sample':y[1]})

In [ ]:
def case10(p=.8,T=.2,gains=(1.,7.)):
 k=np.arange(21);fig,ax=plt.subplots(figsize=(7,2.6));runs=[]
 for K in gains:
  q=p-(1-p)*K
  if abs(q)<1e-14:q=0.0
  if abs(abs(q)-1)<1e-14:q=math.copysign(1.,q)
  yss=K/(1+K);y=yss*(1-q**k)
  direct=[0.]
  for _ in k[1:]:direct.append(p*direct[-1]+(1-p)*K*(1-direct[-1]))
  assert np.allclose(y,direct);ax.plot(k*T,y/yss,'o-',ms=3,label=f'K={K:g}, pole={q:.2f}')
  runs.append({'K':K,'q':q,'yss':yss,'time_constant':(0.0 if q==0 else -T/math.log(abs(q))) if abs(q)<1 else None})
 ax.axhline(1,color='black',ls=':');ax.set(xlabel='Time (s)',ylabel='Output / own steady output');ax.legend(fontsize=9)
 q=runs[0]['q'];settle=(1 if abs(q)<1e-14 else math.ceil(math.log(.02)/math.log(abs(q)))) if abs(q)<1 else None
 return finish(10,fig,{'runs':runs,'two_percent_sample_index_first_run':settle,'settling_time_first_run':settle*T if settle is not None else None})

In [ ]:
def case11(p=.5,K=2.4):
 gains=np.linspace(0,3.3,401);r0=[];rd=[]
 for g in gains:
  r0.append(abs(p-(1-p)*g));rd.append(max(abs(np.roots([1,-p,(1-p)*g]))))
 fig,ax=plt.subplots(figsize=(7,2.6));ax.plot(gains,r0,label='no computation delay');ax.plot(gains,rd,label='one tick late');ax.axhline(1,color='black',ls=':',label='stability boundary');ax.axhline(.85,color='gray',ls='--',label='radius target 0.85');ax.set(xlabel='Proportional gain K',ylabel='Largest pole magnitude');ax.legend(fontsize=8,ncol=2)
 # Independently check Jury conditions against roots across the grid, away from boundaries.
 for g,rad in zip(gains,rd):
  if min(abs(g),abs(g-1/(1-p)))>1e-8:assert (rad<1)==(0<=g<1/(1-p))
 return finish(11,fig,{'K':K,'no_delay_pole':p-(1-p)*K,'delayed_pole_magnitude':float(max(abs(np.roots([1,-p,(1-p)*K])))),'positive_gain_limits':[(1+p)/(1-p),1/(1-p)],'delay_radius_085_gain_limit':.85**2/(1-p),'robust_no_delay_limit_for_p_04_065':1.4/.6})

In [ ]:
CASES={i:globals()[f'case{i:02}'] for i in range(1,12)}

In [ ]:
def run_case(n,**kwargs):return CASES[int(n)](**kwargs)

## Run case 7

Change the keyword arguments below to investigate the assigned variation. Use `help(case07)` or inspect its definition for available parameters.

In [ ]:
result = run_case(7)
from IPython.display import display, Image
display(Image(filename=str(OUT / "case07.png")))
result